# Brent Oil Price: Change-Point Analysis
Birhan Energies — Data Science Consultancy

**Goal:** identify statistically significant regime shifts in Brent crude
prices and match them to major political/economic events, quantifying the
impact for investors, policymakers, and energy companies.


## 1. Setup

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.data_loader import load_brent_prices, clean_prices, add_log_returns
from src.events import get_events_df
from src.change_point import ruptures_change_points, bayesian_single_change_point
from src.analysis import summarize_regime_shift, match_change_points_to_events

plt.rcParams['figure.figsize'] = (12, 5)


## 2. Load data

Place your Brent price CSV in `../data/` and update the filename below.
Expected columns: a date column and a price column (auto-detected).

In [ ]:
DATA_PATH = "../data/brent_prices.csv"  # BrentOilPrices.csv, 1987-05-20 to 2022-11-14

prices = load_brent_prices(DATA_PATH)
prices = clean_prices(prices, method="ffill")
prices.head()


## 3. Exploratory Data Analysis

In [ ]:
fig, ax = plt.subplots()
ax.plot(prices.index, prices["Price"])
ax.set_title("Brent Crude Oil Price")
ax.set_xlabel("Date")
ax.set_ylabel("USD per barrel")
plt.show()


In [ ]:
returns = add_log_returns(prices)

fig, axes = plt.subplots(2, 1, figsize=(12, 8))
axes[0].plot(returns.index, returns["LogReturn"])
axes[0].set_title("Log Returns")
axes[1].hist(returns["LogReturn"], bins=100)
axes[1].set_title("Distribution of Log Returns")
plt.tight_layout()
plt.show()

print(returns["LogReturn"].describe())


## 4. Change-point detection (fast baseline: ruptures / PELT)

Scan for multiple change points before committing to the slower Bayesian
single-change-point model. Adjust `penalty` — lower values find more
change points.

In [ ]:
cp_dates = ruptures_change_points(returns["LogReturn"], model="rbf", penalty=15.0)
print(f"Detected {len(cp_dates)} change point(s):")
for d in cp_dates:
    print(" -", d.date())


In [ ]:
fig, ax = plt.subplots()
ax.plot(prices.index, prices["Price"])
for d in cp_dates:
    ax.axvline(d, color="red", linestyle="--", alpha=0.7)
ax.set_title("Brent Price with Detected Change Points (ruptures/PELT)")
plt.show()


## 5. Bayesian single change-point model (PyMC)

For a focused analysis around one suspected major regime shift, fit the
full Bayesian model to get posterior uncertainty over the change-point
location (not just a point estimate).

In [ ]:
# Example: analyze a subset of the series around a period of interest
# subset = returns.loc["2019-06-01":"2020-12-31", "LogReturn"]
# result = bayesian_single_change_point(subset, draws=1000, tune=1000, chains=2)
# print("Most probable change point:", result["tau_date"])
# print(f"Mean return before: {result['mu_1_mean']:.5f}, after: {result['mu_2_mean']:.5f}")
# print(f"Volatility before: {result['sigma_1_mean']:.5f}, after: {result['sigma_2_mean']:.5f}")


## 6. Match change points to real-world events

In [ ]:
matches = match_change_points_to_events(cp_dates, window_days=14)
matches


## 7. Quantify impact of each regime shift

In [ ]:
impact_summaries = [summarize_regime_shift(prices["Price"], d, window_days=30) for d in cp_dates]
impact_df = pd.DataFrame(impact_summaries)
impact_df


## 8. Insights for stakeholders

Summarize, in plain language, what the detected change points + matched
events imply for:
- **Investors** — timing/magnitude of historical shocks
- **Policymakers** — sensitivity to shock category (supply/demand/geopolitical)
- **Energy companies** — persistence of volatility after a shock, for hedging

_(Fill in after running the analysis on your actual dataset.)_
